In [0]:
SELECT
  *
FROM samples.nyctaxi.trips
limit 10

In [0]:
-- Ejercicio 0.1
WITH promedios(
SELECT
 pickup_zip,
 count(*) conteo,
 round(avg( fare_amount ),2) prom_tarifa,
 round(avg(trip_distance),2) prom_distancia
FROM samples.nyctaxi.trips
GROUP BY pickup_zip
)
SELECT
*
from promedios
where conteo > 100

In [0]:
-- ejercicio 0.2
WITH promedio_general (
  select
    round(avg(fare_amount),2) promedio_general
  from samples.nyctaxi.trips
),
promedio_hora(
  SELECT
    hour(tpep_pickup_datetime) hora,
    round(avg(fare_amount),2) promedio_hora
  from samples.nyctaxi.trips
  GROUP by hour(tpep_pickup_datetime)
)
SELECT
  h.hora,
  h.promedio_hora,
  g.promedio_general,
  round(h.promedio_hora - g.promedio_general,2) diferencia,
  round((h.promedio_hora - g.promedio_general) / g.promedio_general * 100,2)  diferencia_porcetual
from promedio_hora h
cross join promedio_general g



In [0]:
-- ejercicio 0.3
with viajes_validos(
  select
    *
  from samples.nyctaxi.trips
  where
    trip_distance > 0
    and fare_amount > 0
)
select
  pickup_zip,
  count(*) conteo,
  round(avg( fare_amount ),2) prom_tarifa,
  median(fare_amount) mediana_tarifa,
  percentile(fare_amount,0.5) percentil05_tarifa,
  min(fare_amount) min_tarifa,
  max(fare_amount) max_tarifa,
  round(avg(trip_distance),2) prom_distancia,
  median(trip_distance) mediana_distancia,
  percentile(trip_distance,0.5) percentil05_distancia,
  min(trip_distance) min_distancia,
  max(trip_distance) max_distanica
from viajes_validos
group by pickup_zip


In [0]:
-- ejercicio 0.4
select
  pickup_zip,
  tpep_pickup_datetime,
  tpep_dropoff_datetime,
  trip_distance,
  fare_amount,
  row_number() over (order by fare_amount desc) ranking 
from samples.nyctaxi.trips
where
  fare_amount > 0
  and trip_distance > 0
qualify ranking <= 20

In [0]:
-- ejercicio 0.5
with viajes(
    select
      pickup_zip,
      count(*) conteo
    from samples.nyctaxi.trips
    where
      fare_amount > 0
      and trip_distance  > 0
    group by pickup_zip
)
select
  pickup_zip,
  conteo,
  row_number() over(order by conteo desc) row_number,
  rank() over(order by conteo desc) rank,
  dense_rank() over(order by conteo desc) dense_rank
from viajes
order by conteo desc, row_number
-- limit 10


In [0]:
-- ejercicio 0.6
with promedio_general(
select
  pickup_zip,
  fare_amount,
  trip_distance,
  round(avg(fare_amount) over(),2) promedio_general
from samples.nyctaxi.trips
where
  fare_amount > 0
  and trip_distance > 0
)
select
  *,
  round(fare_amount - promedio_general,2) diferencia,
  round((fare_amount-promedio_general)/promedio_general*100,2) diferencia_porcetual
from promedio_general

In [0]:
-- ejercicio 0.7
select
  tpep_pickup_datetime,
  fare_amount,
  pickup_zip,
  count(*) over(partition by pickup_zip) cantidad_viajes_zona,
  round(avg(fare_amount) over(partition by pickup_zip),2) avg_zona
from samples.nyctaxi.trips

In [0]:
-- ejercicio 0.8
select
  tpep_pickup_datetime,
  fare_amount,
  lag(fare_amount) over (order by tpep_pickup_datetime) tarifa_anterior
from samples.nyctaxi.trips
order by tpep_pickup_datetime 

In [0]:
-- ejercicio 0.9
select
  tpep_pickup_datetime,
  fare_amount,
  sum(fare_amount) over(order by tpep_pickup_datetime) running_total
from samples.nyctaxi.trips
order by tpep_pickup_datetime

In [0]:
-- ejercicio 0.10
with viajes_validos(
  select
    *
  from samples.nyctaxi.trips
  where
    trip_distance > 0
    and fare_amount > 0
),
estadisticas_zonas(
  select
    pickup_zip,
    count(*) conteo,
    round(avg( fare_amount ),2) prom_tarifa,
    max(fare_amount) max_tarifa,
    min(fare_amount) min_tarifa,
    round(avg( trip_distance ),2) prom_distancia,
    max(trip_distance) max_distancia,
    min(fare_amount) min_distancia
  from viajes_validos
  group by pickup_zip
)
select
  *,
  row_number() over(order by prom_tarifa desc) ranking_rownumber,
  dense_rank(prom_tarifa) over(order by prom_tarifa desc) ranking_denserank,
  rank(prom_tarifa) over(order by prom_tarifa desc) ranking_rank  
from estadisticas_zonas
qualify ranking_rank <= 10